***

* [总目录](../0_Introduction/0_introduction.ipynb)
* [术语表](../0_Introduction/1_glossary.ipynb)
* [8. 校准](8_0_introduction.ipynb)
    * 上一节： [8.2 第一代校准（1GC）](8_2_1gc.ipynb)
    * 下一节： [8.4 第三代校准（3GC）](8_4_3gc.ipynb)

***


导入标准模块:


In [ ]:
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

try:
    from IPython.display import HTML, display
except ImportError:
    HTML = None
    display = None

STYLE_PATH = Path("../style/course.css")
TOGGLE_PATH = Path("../style/code_toggle.html")

if HTML is not None and display is not None:
    if STYLE_PATH.exists():
        display(HTML(f"<style>{STYLE_PATH.read_text(encoding='utf-8')}</style>"))
    if TOGGLE_PATH.exists():
        display(HTML(TOGGLE_PATH.read_text(encoding="utf-8")))

plt.rcParams["figure.figsize"] = (9, 4.5)
plt.rcParams["axes.grid"] = True
np.set_printoptions(precision=3, suppress=True)

RNG = np.random.default_rng(20260419)


第二代校准（2GC）通常指方向无关自校准。它不再依赖外场校准源，而是直接利用目标场数据本身求解增益，再把改正后的数据送回成像与去卷积步骤，持续更新天空模型。

自校准的威力来自闭环迭代，但它的风险也来自闭环迭代：如果天空模型有系统性错误，求解器会很乐于把模型误差“解释”为增益。理解这一点，是把 2GC 用对而不是用坏的关键。


***


## 8.3 第二代校准（2GC）：方向无关自校准

这一节通过一个最小化的一维成像实验来观察自校准的三个核心特征：

- 用不完整模型做自校准时，图像会改善，但仍可能带偏；
- 当天空模型更完整时，残差和动态范围会显著改善；
- 求解时间间隔过长或过短都会带来代价。


In [ ]:
def baseline_pairs(nant):
    return [(p, q) for p in range(nant) for q in range(p + 1, nant)]


def baseline_u(ant_x_m, hour_angle_h, wavelength_m=0.214):
    pairs = baseline_pairs(len(ant_x_m))
    hour_angle_rad = np.deg2rad(15.0 * hour_angle_h)
    u = np.zeros((hour_angle_h.size, len(pairs)))
    for ti, ha in enumerate(hour_angle_rad):
        for bi, (p, q) in enumerate(pairs):
            u[ti, bi] = (ant_x_m[q] - ant_x_m[p]) * np.sin(ha) / wavelength_m
    return pairs, u


def sky_vis_1d(u, fluxes, ls):
    vis = np.zeros_like(u, dtype=complex)
    for flux, ll in zip(fluxes, ls):
        vis += flux * np.exp(-2j * np.pi * u * ll)
    return vis


def vec_to_cube(vis_vec, pairs, nant):
    nt, nb = vis_vec.shape
    cube = np.zeros((nt, nant, nant), dtype=complex)
    for bi, (p, q) in enumerate(pairs):
        cube[:, p, q] = vis_vec[:, bi]
        cube[:, q, p] = np.conj(vis_vec[:, bi])
    return cube


def cube_to_vec(cube, pairs):
    return np.stack([cube[:, p, q] for p, q in pairs], axis=1)


def apply_gains(model_cube, gains, noise_std=0.0, rng=None):
    data = gains[:, :, None] * model_cube * np.conj(gains[:, None, :])
    if noise_std > 0.0:
        if rng is None:
            rng = np.random.default_rng(0)
        noise = noise_std * (
            rng.normal(size=data.shape) + 1j * rng.normal(size=data.shape)
        )
        data = data + noise
    for p in range(data.shape[1]):
        data[:, p, p] = 0.0
    return data


def solve_gains(data, model, n_iter=30, ref_ant=0, phase_only=False):
    nt, nant, _ = data.shape
    gains = np.ones((nt, nant), dtype=complex)
    eps = 1e-12

    for t in range(nt):
        gt = np.ones(nant, dtype=complex)
        for _ in range(n_iter):
            new = gt.copy()
            for p in range(nant):
                mask = np.ones(nant, dtype=bool)
                mask[p] = False
                num = np.sum(data[t, p, mask] * gt[mask] * np.conj(model[t, p, mask]))
                den = np.sum(
                    np.abs(gt[mask]) ** 2 * np.abs(model[t, p, mask]) ** 2
                ) + eps
                new[p] = num / den

            if phase_only:
                amp = np.maximum(np.abs(new), eps)
                new = new / amp

            ref = new[ref_ant]
            new = new / (ref / max(np.abs(ref), eps))
            gt = new

        gains[t] = gt

    return gains


def dirty_image_1d(u, vis, l_grid):
    u_flat = np.concatenate([u.ravel(), -u.ravel()])
    vis_flat = np.concatenate([vis.ravel(), np.conj(vis.ravel())])
    phase = np.exp(2j * np.pi * u_flat[:, None] * l_grid[None, :])
    return np.real(vis_flat @ phase / vis_flat.size)


def off_source_rms(image, l_grid, source_positions, exclusion=0.005):
    mask = np.ones_like(l_grid, dtype=bool)
    for pos in source_positions:
        mask &= np.abs(l_grid - pos) > exclusion
    return np.sqrt(np.mean(image[mask] ** 2))


def average_in_time(arr, block):
    n = arr.shape[0] // block
    trimmed = arr[: n * block]
    new_shape = (n, block) + arr.shape[1:]
    return trimmed.reshape(new_shape).mean(axis=1)


ant_x = np.array([0.0, 38.0, 102.0, 188.0, 310.0, 462.0])
times_h = np.linspace(-3.0, 3.0, 48)
pairs, u = baseline_u(ant_x, times_h)
nant = ant_x.size

flux_true = np.array([1.0, 0.28, 0.14])
l_true = np.array([0.0, 0.011, -0.021])
vis_true = sky_vis_1d(u, flux_true, l_true)
model_true = vec_to_cube(vis_true, pairs, nant)

ant_phase = np.linspace(0.0, np.pi, nant, endpoint=False)
amp = 1.0 + 0.03 * np.sin(1.1 * times_h[:, None] + ant_phase[None, :])
phase = 0.55 * np.sin(2.4 * times_h[:, None] + 0.6 * ant_phase[None, :])
true_gains = amp * np.exp(1j * phase)

data = apply_gains(model_true, true_gains, noise_std=0.02, rng=RNG)

flux_initial = np.array([1.0])
l_initial = np.array([0.0])
vis_initial = sky_vis_1d(u, flux_initial, l_initial)
model_initial = vec_to_cube(vis_initial, pairs, nant)


### 8.3.1 自校准的闭环：模型、求解与再成像 <a id='cal:sec:selfcal'></a>

先从一个故意不完整的天空模型开始。这里我们只保留视场中心的亮源，而忽略两个较弱的离轴源。这样做并不合理，但它非常适合说明“错误模型也能让图像变好，却不一定让结果更真实”这一点。


In [ ]:
gains_phase_only = solve_gains(data, model_initial, n_iter=30, ref_ant=0, phase_only=True)
data_phase_only = data / (
    gains_phase_only[:, :, None] * np.conj(gains_phase_only[:, None, :]) + 1e-12
)

gains_full = solve_gains(data, model_true, n_iter=30, ref_ant=0, phase_only=False)
data_full = data / (
    gains_full[:, :, None] * np.conj(gains_full[:, None, :]) + 1e-12
)

l_grid = np.linspace(-0.04, 0.04, 700)
image_raw = dirty_image_1d(u, cube_to_vec(data, pairs), l_grid)
image_phase_only = dirty_image_1d(u, cube_to_vec(data_phase_only, pairs), l_grid)
image_full = dirty_image_1d(u, cube_to_vec(data_full, pairs), l_grid)

true_sky_image = np.zeros_like(l_grid)
for flux, ll in zip(flux_true, l_true):
    idx = np.argmin(np.abs(l_grid - ll))
    true_sky_image[idx] = flux

rms_raw = off_source_rms(image_raw, l_grid, l_true)
rms_phase = off_source_rms(image_phase_only, l_grid, l_true)
rms_full = off_source_rms(image_full, l_grid, l_true)
peak_raw = image_raw.max()
peak_phase = image_phase_only.max()
peak_full = image_full.max()

fig, axes = plt.subplots(3, 1, figsize=(10, 9), sharex=True)

for ax, img, title, color in [
    (axes[0], image_raw, "未自校准", "tab:red"),
    (axes[1], image_phase_only, "相位自校准（模型不完整）", "tab:orange"),
    (axes[2], image_full, "幅相自校准（模型更完整）", "tab:blue"),
]:
    ax.plot(l_grid, img, color=color, lw=1.6)
    for pos in l_true:
        ax.axvline(pos, color="black", ls=":", alpha=0.6)
    ax.set_ylabel("dirty image")
    ax.set_title(title)

axes[2].set_xlabel("direction cosine l")
plt.tight_layout()

print(f"离源 RMS：原始 {rms_raw:.4f}，相位自校后 {rms_phase:.4f}，完整自校后 {rms_full:.4f}")
print(f"峰值响应：原始 {peak_raw:.4f}，相位自校后 {peak_phase:.4f}，完整自校后 {peak_full:.4f}")


这个例子里，自校准确实让图像逐步变干净了，但两轮迭代的物理含义并不相同：

- **第一轮相位自校准**主要利用亮核建立相位参考，因此中心结构显著改善；
- **第二轮更完整模型**把离轴源也纳入可见度预测中，于是求解器不再需要把它们“错误地吸收入增益”，残差才会进一步下降。

这也是为什么实践中常说“自校准不是魔法，它依赖好的 sky model”。


### 8.3.2 解间隔（solution interval）的权衡

自校准还有一个核心参数是解间隔。间隔太长，会把快速相位变化平均掉；间隔太短，则可能因为信噪比不足而得到噪声主导的解。下面的实验只模拟前一类风险，即相位波动快于求解间隔时会发生什么。


In [ ]:
intervals = [1, 2, 4, 8]
residual_rms = []

for block in intervals:
    if times_h.size // block < 2:
        continue
    data_avg = average_in_time(data, block)
    model_avg = average_in_time(model_true, block)
    gains_avg = solve_gains(data_avg, model_avg, n_iter=25, ref_ant=0, phase_only=True)
    gains_interp = np.repeat(gains_avg, block, axis=0)[: times_h.size]
    data_corr = data / (
        gains_interp[:, :, None] * np.conj(gains_interp[:, None, :]) + 1e-12
    )
    residual = cube_to_vec(data_corr - model_true, pairs)
    residual_rms.append(np.sqrt(np.mean(np.abs(residual) ** 2)))

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(intervals[: len(residual_rms)], residual_rms, marker="o", lw=2.0, color="tab:purple")
ax.set_xlabel("solution interval [number of time samples]")
ax.set_ylabel("residual visibility RMS")
ax.set_title("Too-long solution intervals fail to track rapid phase changes")
plt.tight_layout()


真正的数据处理中，这条曲线往往是一个两端都不理想的“碗形”关系：太短的间隔噪声太大，太长的间隔又跟不上真实变化。因此，合适的解间隔总要结合阵列灵敏度、目标场亮度、频段和天气条件来选。


### 8.3.3 自校准、混合成图与失败模式

经典的 *hybrid mapping* 可以概括成一个循环：

1. 用当前数据做成像和去卷积；
2. 从图像中提取或更新天空模型；
3. 用该模型求解增益；
4. 改正数据并返回第 1 步。

这个闭环在以下情况下尤其容易出问题：

- 初始模型过于贫乏，导致解把扩展结构“吃掉”；
- 场内总信噪比不足，却强行做过密的解；
- 幅相自校准开始得太早，导致绝对通量标尺漂移；
- 把方向依赖残差误当成方向无关相位误差处理。

因此，一个稳健的经验做法通常是：先做 1GC，再做相位自校准，确认模型和残差稳定后，再谨慎进入幅相自校准。若此时视场仍表现出明显位置相关残差，就该考虑 3GC 了。
